# Lab 2 - Curate: dedup, toxicity, auto-label, human review, gap-fill

| Step | Technique |
|---|---|
| Exact dedup | hash of normalized text |
| Near-dup | **MinHash + LSH** on word 3-grams (no embedding model) |
| Toxicity | **gpt-oss-safeguard-20b** + a plain-text policy you can edit |
| Auto-label | **gpt-oss-20b** extracts labels + confidence, calibrated on a human gold set |
| Human review | in-notebook review queue for low-confidence and edge cases |
| Gap-fill | generate new requests for under-covered segments (same technique as Lab 1) |

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from octank import config, state
S = config.get_settings("attendee")
ST = state.load(S)
print(f"region={S.region}  bucket={S.bucket}  prefix={S.prefix}  run={S.run_name}")
print(f"role={S.role_arn}")

In [ ]:
from octank import catalog, dedup, safety, label, review, profile, gold
from octank.pipeline import ATTENDEE, Run
run = Run(S, ATTENDEE)
raw = run.load_raw()
cat = catalog.load_clean()
print(len(raw), "raw rows")

## 1. De-duplication

**MinHash** turns each message into a 128-number signature; two signatures agree on a fraction of
positions roughly equal to the **Jaccard similarity** of the messages' word 3-grams. **LSH**
buckets signatures so we only compare likely matches. Try a few thresholds.

In [ ]:
exact = dedup.exact_duplicates(raw)
print("exact duplicates:", int(exact.sum()))
d1 = raw[~exact]
for t in (0.3, 0.5, 0.7, 0.9):
    print(f"near-dup threshold {t}: {len(dedup.near_duplicates(d1, threshold=t))} pairs")

In [ ]:
pairs = dedup.near_duplicates(d1, threshold=ATTENDEE.near_dup_threshold)
msg = d1.set_index("request_id")["message"]
for p in pairs.head(4).itertuples():
    print(f"J~{p.jaccard:.2f}\n  keep: {msg[p.keep_id][:110]}\n  drop: {msg[p.dup_id][:110]}\n")
deduped = d1[~d1["request_id"].isin(pairs["dup_id"])]
print(len(raw), "->", len(deduped))

## 2. Toxicity filter

`gpt-oss-safeguard-20b` is an open-weight **safety classifier**: you give it a *policy* in plain
language and it classifies content against it, with a rationale. Open the policy, read it, and
try editing it (e.g. add a rule) and re-running.

In [ ]:
policy = safety.load_policy()
print(policy[:1500])

In [ ]:
%%time
screened = run.safety(deduped, policy=policy)
print(screened["safety_decision"].value_counts())
screened[screened["safety_violation"] == 1][["message", "safety_category", "safety_confidence",
                                             "safety_rationale"]].head(10)

In [ ]:
kept = screened[screened["safety_decision"] != "drop"]
print(len(deduped), "->", len(kept))

## 3. Auto-labeling - calibrate first, then label at scale

The free-text message is the source of truth; form fields are often missing or wrong.
`gpt-oss-20b` extracts structured labels and a **confidence**. Before trusting it at scale we
measure it on the **gold set** (draft labels from `gpt-oss-120b`, corrected by humans) and pick
the confidence threshold for auto-accept vs. human review.

In [ ]:
g = gold.load_gold()
print(len(g), "gold rows;", g["label_source"].value_counts().to_dict())
pred = label.label_df(g[["request_id", "message"]], cache=run.cache("gold_pred"),
                      region=S.region)
pred.columns = [c.replace("lbl_", "") for c in pred.columns]
print("field accuracy vs gold (%):", label.field_accuracy(pred, g))
label.calibration_curve(pred, g)

In [ ]:
REVIEW_THRESHOLD = 0.7     # <- pick from the curve above: accuracy vs. how much you send to humans
run.cfg.review_threshold = REVIEW_THRESHOLD
labeled = run.label(kept)
labeled["review_reason"].value_counts()

## 4. Human review

Rows that need a person: unclear intent, form vs. text disagreements, no budget anywhere,
borderline safety and low confidence. Review **~15 items** (approve / edit / reject). Anything you
don't get to is resolved by the default policy, so the pipeline always completes.

In [ ]:
queue = labeled[labeled["review_reason"] != ""].sort_values("review_reason")
q = review.ReviewQueue(queue.head(15), save_path=run.path("review_decisions.parquet"))
q.show()

In [ ]:
human = q.decisions()
print(len(human), "human decisions")
curated, decisions = run.resolve(labeled, human=human)
curated = run.finalize(curated, cat)
print(len(labeled), "->", len(curated), "rows;", decisions["decision"].value_counts().to_dict())

## 5. Did curation work? (scored against the hidden answer key)

In [ ]:
run.report(raw, curated)

## 6. Synthetic gap-fill

Generate new requests for segments below 2% of the data, using the same generator - then send
them through the **same** safety, labeling and finalize steps.

In [ ]:
%%time
final, gaps = run.gapfill(curated, cat)
print(len(curated), "->", len(final), "rows")
before, after = profile.skew_report(curated), profile.skew_report(final)
print("occasion skew before:", before, "\nafter: ", after)
profile.plot_overview(final, "After curation + gap-fill");

In [ ]:
final.to_parquet(run.path("final.parquet"), index=False)
state.update(S, final_rows=len(final))